In [12]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import time
import pickle

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

RANDOM_STATE = 15

df = pd.read_csv("df_class_ordinal.csv")
df = df[df['crash_sev_ordinal'] != 1].copy()
df['target4'] = df['crash_sev_ordinal'].map({0:0, 2:1, 3:2, 4:3})


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 209262 entries, 0 to 210586
Data columns (total 25 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   crash_speed_limit                       209262 non-null  float64
 1   road_constr_zone_fl                     209262 non-null  bool   
 2   latitude                                209262 non-null  float64
 3   longitude                               209262 non-null  float64
 4   onsys_fl                                209262 non-null  int64  
 5   private_dr_fl                           209262 non-null  int64  
 6   crash_speed_limit_missing               209262 non-null  int64  
 7   collision_type_grouped_head_on          209262 non-null  bool   
 8   collision_type_grouped_other            209262 non-null  bool   
 9   collision_type_grouped_parking_related  209262 non-null  bool   
 10  collision_type_grouped_rear_end         209262 no

In [15]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 209262 entries, 0 to 210586
Data columns (total 13 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   crash_speed_limit                       209262 non-null  float64
 1   num_units                               209262 non-null  int64  
 2   collision_type_grouped_head_on          209262 non-null  bool   
 3   collision_type_grouped_other            209262 non-null  bool   
 4   collision_type_grouped_parking_related  209262 non-null  bool   
 5   collision_type_grouped_rear_end         209262 non-null  bool   
 6   collision_type_grouped_sideswipe        209262 non-null  bool   
 7   collision_type_grouped_single_vehicle   209262 non-null  bool   
 8   collision_type_grouped_turning          209262 non-null  bool   
 9   day_of_week                             209262 non-null  int64  
 10  time_bucket_midday                      209262 no

In [13]:
selected_features = ['crash_speed_limit', 'num_units', 'time_bucket', 'collision_type_grouped_head_on', 'collision_type_grouped_other', 'collision_type_grouped_parking_related', 'collision_type_grouped_rear_end', 'collision_type_grouped_sideswipe', 'collision_type_grouped_single_vehicle', 'collision_type_grouped_turning', 'day_of_week']
X = df[selected_features].copy()
y = df['crash_sev_ordinal'].replace({0:0, 2:1, 3:2, 4:3})

X = pd.get_dummies(X, columns=['time_bucket'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

def evaluate(name, y_true, y_pred, duration=0):
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    wf1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    prec_per = precision_score(y_true, y_pred, average=None, zero_division=0)
    rec_per = recall_score(y_true, y_pred, average=None, zero_division=0)
    f1_per = f1_score(y_true, y_pred, average=None, zero_division=0)

    print(f"--- {name} ---")
    print(f"Time: {duration:.2f}s | Accuracy: {acc:.4f} | Macro F1: {mf1:.4f} | Weighted F1: {wf1:.4f}")
    print(f"Fatal: Precision={prec_per[3]:.3f}, Recall={rec_per[3]:.3f}, F1={f1_per[3]:.3f}")

    return {
        'accuracy': acc,
        'macro_f1': mf1,
        'weighted_f1': wf1,
        'Fatal_precision': prec_per[3],
        'Fatal_recall': rec_per[3],
        'Fatal_f1': f1_per[3]
    }

baseline_results = {}

t = time.time()
lr = LogisticRegression(max_iter=1000, class_weight='balanced',
                       random_state=RANDOM_STATE, n_jobs=-1)
lr.fit(X_train_s, y_train)
baseline_results['Logistic Regression'] = evaluate(
    'Logistic Regression', y_test, lr.predict(X_test_s), time.time()-t
)

t = time.time()
dt = DecisionTreeClassifier(max_depth=10, class_weight='balanced',
                           random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
baseline_results['Decision Tree'] = evaluate(
    'Decision Tree', y_test, dt.predict(X_test), time.time()-t
)

t = time.time()
rf = RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced',
                           random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
baseline_results['Random Forest'] = evaluate(
    'Random Forest', y_test, rf.predict(X_test), time.time()-t
)

t = time.time()
gb = GradientBoostingClassifier(n_estimators=50, max_depth=5, random_state=RANDOM_STATE)
gb.fit(X_train, y_train)
baseline_results['Gradient Boosting'] = evaluate(
    'Gradient Boosting', y_test, gb.predict(X_test), time.time()-t
)

t = time.time()
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                   random_state=RANDOM_STATE, n_jobs=-1, eval_metric='mlogloss')
xgb.fit(X_train, y_train)
baseline_results['XGBoost'] = evaluate(
    'XGBoost', y_test, xgb.predict(X_test), time.time()-t
)

improv_results = {}

t = time.time()
rf_base = RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced',
                                  random_state=RANDOM_STATE, n_jobs=-1)
rf_base.fit(X_train, y_train)
improv_results['Baseline RF'] = evaluate(
    'Baseline RF', y_test, rf_base.predict(X_test), time.time()-t
)

sm = SMOTE(random_state=RANDOM_STATE)
X_sm, y_sm = sm.fit_resample(X_train, y_train)

t = time.time()
rf_smote = RandomForestClassifier(n_estimators=200, max_depth=15,
                                   random_state=RANDOM_STATE, n_jobs=-1)
rf_smote.fit(X_sm, y_sm)
improv_results['RF + SMOTE'] = evaluate(
    'RF + SMOTE', y_test, rf_smote.predict(X_test), time.time()-t
)

n = len(y_train); k = 4
class_counts = np.bincount(y_train)
class_weights_inv = n / (k * class_counts)
sample_weight = np.array([class_weights_inv[c] for c in y_train])

t = time.time()
xgb_cw = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                        random_state=RANDOM_STATE, n_jobs=-1, eval_metric='mlogloss')
xgb_cw.fit(X_train, y_train, sample_weight=sample_weight)
improv_results['XGB + weights'] = evaluate(
    'XGB + weights', y_test, xgb_cw.predict(X_test), time.time()-t
)

y_proba = xgb_cw.predict_proba(X_test)
yp = y_proba.copy()
yp[:, 3] *= 1.5
yp = yp / yp.sum(axis=1, keepdims=True)
y_pred_th = yp.argmax(axis=1)

improv_results['XGB threshold'] = evaluate(
    'XGB threshold', y_test, y_pred_th, 0
)

t = time.time()
rf_tuned = RandomForestClassifier(
    n_estimators=300, max_depth=25,
    min_samples_leaf=5, min_samples_split=10,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_tuned.fit(X_train, y_train)
improv_results['Tuned RF'] = evaluate(
    'Tuned RF', y_test, rf_tuned.predict(X_test), time.time()-t
)

rf_cv = RandomForestClassifier(
    n_estimators=100, max_depth=25,
    min_samples_leaf=5, min_samples_split=10,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE, n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(rf_cv, X_train, y_train, cv=cv,
                            scoring='f1_macro', n_jobs=-1)

print(f"Macro F1 per fold: {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean: {cv_scores.mean():.4f}")
print(f"Std:  {cv_scores.std():.4f}")


--- Logistic Regression ---
Time: 11.58s | Accuracy: 0.2639 | Macro F1: 0.2247 | Weighted F1: 0.2808
Fatal: Precision=0.056, Recall=0.546, F1=0.101
--- Decision Tree ---
Time: 0.50s | Accuracy: 0.2918 | Macro F1: 0.2463 | Weighted F1: 0.3110
Fatal: Precision=0.055, Recall=0.411, F1=0.098
--- Random Forest ---
Time: 11.25s | Accuracy: 0.3053 | Macro F1: 0.2607 | Weighted F1: 0.3346
Fatal: Precision=0.056, Recall=0.365, F1=0.096
--- Gradient Boosting ---
Time: 73.99s | Accuracy: 0.5159 | Macro F1: 0.2123 | Weighted F1: 0.3833
Fatal: Precision=0.000, Recall=0.000, F1=0.000
--- XGBoost ---
Time: 10.85s | Accuracy: 0.5158 | Macro F1: 0.2140 | Weighted F1: 0.3849
Fatal: Precision=0.000, Recall=0.000, F1=0.000
--- Baseline RF ---
Time: 10.81s | Accuracy: 0.3053 | Macro F1: 0.2607 | Weighted F1: 0.3346
Fatal: Precision=0.056, Recall=0.365, F1=0.096
--- RF + SMOTE ---
Time: 33.65s | Accuracy: 0.2922 | Macro F1: 0.2549 | Weighted F1: 0.3178
Fatal: Precision=0.056, Recall=0.379, F1=0.098
--- XGB 

In [16]:

with open("clf_agent.pkl", 'wb') as f:
    pickle.dump(rf_tuned, f)

print('Model saved')

Model saved


## REGRESSION

In [14]:
X_ohe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210587 entries, 0 to 210586
Data columns (total 13 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   crash_speed_limit                       210587 non-null  float64
 1   num_units                               210587 non-null  int64  
 2   collision_type_grouped_head_on          210587 non-null  bool   
 3   collision_type_grouped_other            210587 non-null  bool   
 4   collision_type_grouped_parking_related  210587 non-null  bool   
 5   collision_type_grouped_rear_end         210587 non-null  bool   
 6   collision_type_grouped_sideswipe        210587 non-null  bool   
 7   collision_type_grouped_single_vehicle   210587 non-null  bool   
 8   collision_type_grouped_turning          210587 non-null  bool   
 9   day_of_week                             210587 non-null  int64  
 10  time_bucket_midday                      2105

In [8]:
## imports
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import make_scorer

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from statsmodels.discrete.count_model import ZeroInflatedPoisson

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

## metrics
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

def evaluate(name, y_true, y_pred):
    print(f"\n{name}")
    print(regression_metrics(y_true, y_pred))

## data prep
df = pd.read_csv("df_reg_cleaned.csv")

features = [
    'crash_speed_limit', 'num_units', 'time_bucket',
    'collision_type_grouped_head_on',
    'collision_type_grouped_other',
    'collision_type_grouped_parking_related',
    'collision_type_grouped_rear_end',
    'collision_type_grouped_sideswipe',
    'collision_type_grouped_single_vehicle',
    'collision_type_grouped_turning',
    'day_of_week'
]

y = df["injury_count"]
y_log = np.log1p(y)

X_raw = df[features].copy()

X_ohe = pd.get_dummies(X_raw, drop_first=True)

df_le = df[features + ["injury_count"]].copy()
le = LabelEncoder()
df_le["time_bucket"] = le.fit_transform(df_le["time_bucket"])

X_le = df_le.drop(columns=["injury_count"])
X_le = X_le.apply(pd.to_numeric, errors="coerce").astype(float)

## Train/Test splits
X_train_ohe, X_test_ohe, y_train, y_test = train_test_split(
    X_ohe, y, test_size=0.2, random_state=42
)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

X_train_le, X_test_le, _, _ = train_test_split(
    X_le, y, test_size=0.2, random_state=42
)

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

### vif branch
def compute_vif(X):
    vif = pd.DataFrame()
    vif["feature"] = X.columns
    vif["VIF"] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]
    return vif.sort_values(by="VIF", ascending=False)

print(compute_vif(X_train_le))

X_train_vif = X_train_le.copy()
X_test_vif = X_test_le.copy()

print(compute_vif(X_train_vif))

## linear + Poisson
lr = LinearRegression()
lr.fit(X_train_ohe, y_train)
evaluate("Linear (Raw)", y_test, lr.predict(X_test_ohe))

lr.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(lr.predict(X_test_ohe))
evaluate("Linear (Log)", y_test, y_pred)

poisson = PoissonRegressor(alpha=1.0)

poisson.fit(X_train_ohe, y_train)
evaluate("Poisson (Raw)", y_test, poisson.predict(X_test_ohe))

poisson.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(poisson.predict(X_test_ohe))
evaluate("Poisson (Log)", y_test, y_pred)

## tree models
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_ohe, y_train)
evaluate("Random Forest", y_test, rf.predict(X_test_ohe))

gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train_ohe, y_train)
evaluate("Gradient Boosting", y_test, gb.predict(X_test_ohe))

## xgboost
xgb = XGBRegressor(objective="reg:squarederror", random_state=42, verbosity=0)
xgb.fit(X_train_ohe, y_train)
evaluate("XGBoost (Raw)", y_test, xgb.predict(X_test_ohe))

xgb.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(xgb.predict(X_test_ohe))
evaluate("XGBoost (Log)", y_test, y_pred)

## xgboost tuned
param_grid = {
    "n_estimators": [100, 150],
    "max_depth": [3, 4],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "gamma": [0]
}

grid = GridSearchCV(
    estimator=XGBRegressor(objective="reg:squarederror", random_state=42),
    param_grid=param_grid,
    scoring=make_scorer(r2_score),
    cv=5,
    n_jobs=-1
)

grid.fit(X_train_ohe, y_train)

best_xgb = grid.best_estimator_
print("Best params:", grid.best_params_)

evaluate("XGBoost Tuned", y_test, best_xgb.predict(X_test_ohe))

## zip
X_train_sm = sm.add_constant(X_train_vif)
X_test_sm = sm.add_constant(X_test_vif)

zip_model = ZeroInflatedPoisson(
    endog=y_train,
    exog=X_train_sm,
    exog_infl=X_train_sm,
    inflation="logit"
)

zip_res = zip_model.fit(maxiter=500, method="bfgs")

y_pred = zip_res.predict(exog=X_test_sm, exog_infl=X_test_sm)
evaluate("ZIP Model (VIF Features)", y_test, y_pred)

## catboost
X_cat = X_train_raw.copy()
X_cat_test = X_test_raw.copy()

cat_cols = X_cat.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

for col in cat_cols:
    X_cat[col] = X_cat[col].astype(str)
    X_cat_test[col] = X_cat_test[col].astype(str)

cat_features = [X_cat.columns.get_loc(col) for col in cat_cols]

cat_model = CatBoostRegressor(
    loss_function="RMSE",
    verbose=0,
    random_seed=42
)

cat_model.fit(X_cat, y_train_raw, cat_features=cat_features)

evaluate("CatBoost", y_test_raw, cat_model.predict(X_cat_test))

## Model Comparisons
comparison_results = []

def add_result(name, y_true, y_pred, model=None, X=None):
    metrics = regression_metrics(y_true, y_pred)
    
    result = {
        "Model": name,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "AIC": None,
        "BIC": None
    }
    
    if model is not None and X is not None:
        try:
            n = len(y_true)
            residuals = y_true - y_pred
            sse = np.sum(residuals**2)
            k = X.shape[1]

            aic = n * np.log(sse/n) + 2 * k
            bic = n * np.log(sse/n) + k * np.log(n)

            result["AIC"] = aic
            result["BIC"] = bic
        except:
            pass

    comparison_results.append(result)

add_result("Linear (Raw)", y_test, lr.predict(X_test_ohe), lr, X_test_ohe)

lr.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(lr.predict(X_test_ohe))
add_result("Linear (Log)", y_test, y_pred, lr, X_test_ohe)

poisson.fit(X_train_ohe, y_train)
add_result("Poisson (Raw)", y_test, poisson.predict(X_test_ohe), poisson, X_test_ohe)

poisson.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(poisson.predict(X_test_ohe))
add_result("Poisson (Log)", y_test, y_pred, poisson, X_test_ohe)

add_result("Random Forest", y_test, rf.predict(X_test_ohe))
add_result("Gradient Boosting", y_test, gb.predict(X_test_ohe))
add_result("XGBoost (Raw)", y_test, xgb.predict(X_test_ohe))

xgb.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(xgb.predict(X_test_ohe))
add_result("XGBoost (Log)", y_test, y_pred)

add_result("XGBoost Tuned", y_test, best_xgb.predict(X_test_ohe))

zip_aic = zip_res.aic
zip_bic = zip_res.bic
y_pred = zip_res.predict(exog=X_test_sm, exog_infl=X_test_sm)

metrics = regression_metrics(y_test, y_pred)
comparison_results.append({
    "Model": "ZIP Model (VIF Features)",
    "MAE": metrics["MAE"],
    "RMSE": metrics["RMSE"],
    "R2": metrics["R2"],
    "AIC": zip_aic,
    "BIC": zip_bic
})

add_result("CatBoost", y_test_raw, cat_model.predict(X_cat_test))

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values(by="R2", ascending=False)
comparison_df

                                   feature       VIF
0                        crash_speed_limit  6.951839
1                                num_units  5.682773
10                             day_of_week  3.075971
2                              time_bucket  2.871775
8    collision_type_grouped_single_vehicle  2.041226
6          collision_type_grouped_rear_end  1.773348
4             collision_type_grouped_other  1.728331
3           collision_type_grouped_head_on  1.443253
7         collision_type_grouped_sideswipe  1.358663
9           collision_type_grouped_turning  1.154560
5   collision_type_grouped_parking_related  1.000274
                                   feature       VIF
0                        crash_speed_limit  6.951839
1                                num_units  5.682773
10                             day_of_week  3.075971
2                              time_bucket  2.871775
8    collision_type_grouped_single_vehicle  2.041226
6          collision_type_grouped_rear_end  1.

,Model,MAE,RMSE,R2,AIC,BIC
8,XGBoost Tuned,0.718911,1.005406,0.030609,NaN,NaN
5,Gradient Boosting,0.719569,1.005528,0.030374,NaN,NaN
10,CatBoost,0.717732,1.005792,0.029865,NaN,NaN
9,ZIP Model (VIF Features),0.728638,1.009285,0.023113,386516.966596,386757.794765
2,Poisson (Raw),0.740494,1.018129,0.005918,1539.476244,1651.903240
4,Random Forest,0.728386,1.021173,-0.000035,NaN,NaN
7,XGBoost (Log),0.717217,1.028435,-0.014308,NaN,NaN
1,Linear (Log),0.729533,1.031158,-0.019687,2610.562781,2722.989777
3,Poisson (Log),0.739099,1.040596,-0.038438,3378.047008,3490.474004
6,XGBoost (Raw),0.723402,1.057090,-0.071618,NaN,NaN


In [9]:
## save model
with open('reg_agent.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)

print("model saved.")

model saved.
